In [4]:
import pandas as pd
import numpy as np
import os
from scipy import stats

from sklearn.datasets import make_regression, make_classification
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Regression Models
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score

# Classification Models
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. UTILITY: GENERATE & READ DATA FILES
# ==========================================
def create_dummy_files():
    """Generates synthetic data and saves them to CSV, JSON, and Excel."""
    print("--- Generating Dummy Files ---")
    
    # Dataset 1: Regression (saved as CSV)
    X_reg, y_reg = make_regression(n_samples=1000, n_features=10, noise=0.1, random_state=42)
    df_reg = pd.DataFrame(X_reg, columns=[f'feature_{i}' for i in range(10)])
    df_reg['target'] = y_reg
    # Inject some deliberate messiness
    df_reg.loc[10:15, 'feature_0'] = np.nan # Missing values
    df_reg = pd.concat([df_reg, df_reg.iloc[0:5]]) # Duplicates
    df_reg.loc[20, 'feature_1'] = 9999.0 # Outlier
    df_reg.to_csv("regression_data.csv", index=False)
    
    # Dataset 2: Classification (saved as JSON)
    X_clf, y_clf = make_classification(n_samples=1000, n_features=10, random_state=42)
    df_clf = pd.DataFrame(X_clf, columns=[f'feature_{i}' for i in range(10)])
    df_clf['target'] = y_clf
    df_clf.to_json("classification_data.json", orient='records')

    # Dataset 3: Just to demonstrate Excel reading
    df_clf.to_excel("sample_data.xlsx", index=False)
    print("Files created: regression_data.csv, classification_data.json, sample_data.xlsx\n")

def load_data(filepath):
    """Reads data dynamically based on the file extension."""
    ext = os.path.splitext(filepath)[1].lower()
    if ext == '.csv':
        return pd.read_csv(filepath)
    elif ext == '.xlsx' or ext == '.xls':
        return pd.read_excel(filepath)
    elif ext == '.json':
        return pd.read_json(filepath)
    else:
        raise ValueError(f"Unsupported file extension: {ext}")

# ==========================================
# 2. PREPROCESSING WORKFLOW
# ==========================================
def clean_data(df, target_col):
    """Handles duplicates, missing values, and outliers."""
    print(f"Original shape: {df.shape}")
    
    # 1. Duplicate Data Testing
    duplicates = df.duplicated().sum()
    print(f"Found {duplicates} duplicates. Removing...")
    df = df.drop_duplicates()
    
    # 2. Missing Value Checking
    missing = df.isnull().sum().sum()
    print(f"Found {missing} missing values. Imputing with median...")
    # Fill missing numeric values with the column median
    df = df.fillna(df.median())
    
    # 3. Outlier Checking (Using Z-Score on features)
    features = df.drop(columns=[target_col])
    # Keep rows where all feature Z-scores are between -3 and 3
    z_scores = np.abs(stats.zscore(features))
    outlier_mask = (z_scores < 3).all(axis=1)
    outliers_removed = (~outlier_mask).sum()
    print(f"Found {outliers_removed} outliers (Z-score > 3). Removing...")
    
    df_cleaned = df[outlier_mask]
    print(f"Cleaned shape: {df_cleaned.shape}\n")
    
    return df_cleaned.drop(columns=[target_col]), df_cleaned[target_col]

# ==========================================
# 3. REGRESSION PIPELINE (Compare 3 Regularizations)
# ==========================================
def run_regression_pipeline(X, y):
    print("=== REGRESSION PIPELINE ===")
    
    # Train-Test Split (80/20)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Set up K-Fold Cross Validation
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    
    # Models to compare
    models = {
        "Ridge (L2)": (Ridge(), {'model__alpha': [0.1, 1.0, 10.0]}),
        "Lasso (L1)": (Lasso(), {'model__alpha': [0.1, 1.0, 10.0]}),
        "ElasticNet (L1+L2)": (ElasticNet(), {'model__alpha': [0.1, 1.0], 'model__l1_ratio': [0.2, 0.5, 0.8]})
    }
    
    best_overall_model = None
    best_overall_score = float('inf')
    best_model_name = ""

    for name, (model, params) in models.items():
        # Pipeline prevents data leakage during CV
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('model', model)
        ])
        
        # Cross-validation with hyperparameter tuning
        grid = GridSearchCV(pipeline, params, cv=kf, scoring='neg_mean_squared_error')
        grid.fit(X_train, y_train)
        
        rmse = np.sqrt(-grid.best_score_)
        print(f"{name} Best CV RMSE: {rmse:.4f} | Params: {grid.best_params_}")
        
        if rmse < best_overall_score:
            best_overall_score = rmse
            best_overall_model = grid.best_estimator_
            best_model_name = name

    # Final Evaluation on unseen test data
    print(f"\nEvaluating Best Regression Model ({best_model_name}) on Holdout Test Set:")
    y_pred = best_overall_model.predict(X_test)
    print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
    print(f"Test R2 Score: {r2_score(y_test, y_pred):.4f}\n")

# ==========================================
# 4. CLASSIFICATION PIPELINE (Logistic Regression Penalties)
# ==========================================
def run_classification_pipeline(X, y):
    print("=== CLASSIFICATION PIPELINE ===")
    
    # Train-Test Split (Stratified to maintain class balance)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    
    # Stratified K-Fold for Classification
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    # Logistic Regression with different regularization methods
    # solver='saga' supports all penalties (l1, l2, elasticnet)
    models = {
        "Logistic (L2 Ridge)": (LogisticRegression(solver='saga', penalty='l2', max_iter=1000), 
                                {'model__C': [0.1, 1.0, 10.0]}),
        "Logistic (L1 Lasso)": (LogisticRegression(solver='saga', penalty='l1', max_iter=1000), 
                                {'model__C': [0.1, 1.0, 10.0]}),
        "Logistic (ElasticNet)": (LogisticRegression(solver='saga', penalty='elasticnet', max_iter=1000), 
                                  {'model__C': [0.1, 1.0], 'model__l1_ratio': [0.2, 0.5, 0.8]})
    }
    
    best_overall_model = None
    best_overall_score = 0
    best_model_name = ""

    for name, (model, params) in models.items():
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('model', model)
        ])
        
        grid = GridSearchCV(pipeline, params, cv=skf, scoring='accuracy')
        grid.fit(X_train, y_train)
        
        print(f"{name} Best CV Accuracy: {grid.best_score_:.4f} | Params: {grid.best_params_}")
        
        if grid.best_score_ > best_overall_score:
            best_overall_score = grid.best_score_
            best_overall_model = grid.best_estimator_
            best_model_name = name

    print(f"\nEvaluating Best Classification Model ({best_model_name}) on Holdout Test Set:")
    y_pred = best_overall_model.predict(X_test)
    print(classification_report(y_test, y_pred))

# ==========================================
# 5. MAIN EXECUTION BLOCK
# ==========================================
if __name__ == "__main__":
    # 1. Create files so we have something to read
    create_dummy_files()
    
    # 2. Process Regression Data (CSV)
    print("LOADING CSV FILE...")
    df_reg = load_data("regression_data.csv")
    X_reg, y_reg = clean_data(df_reg, target_col='target')
    run_regression_pipeline(X_reg, y_reg)
    
    # 3. Process Classification Data (JSON)
    # (We could also use "sample_data.xlsx" here)
    print("LOADING JSON FILE...")
    df_clf = load_data("classification_data.json")
    X_clf, y_clf = clean_data(df_clf, target_col='target')
    run_classification_pipeline(X_clf, y_clf)

--- Generating Dummy Files ---
Files created: regression_data.csv, classification_data.json, sample_data.xlsx

LOADING CSV FILE...
Original shape: (1005, 11)
Found 5 duplicates. Removing...
Found 6 missing values. Imputing with median...
Found 24 outliers (Z-score > 3). Removing...
Cleaned shape: (976, 11)

=== REGRESSION PIPELINE ===
Ridge (L2) Best CV RMSE: 1.8812 | Params: {'model__alpha': 0.1}
Lasso (L1) Best CV RMSE: 1.9112 | Params: {'model__alpha': 0.1}
ElasticNet (L1+L2) Best CV RMSE: 3.5332 | Params: {'model__alpha': 0.1, 'model__l1_ratio': 0.8}

Evaluating Best Regression Model (Ridge (L2)) on Holdout Test Set:
Test RMSE: 0.1810
Test R2 Score: 1.0000

LOADING JSON FILE...
Original shape: (1000, 11)
Found 0 duplicates. Removing...
Found 0 missing values. Imputing with median...
Found 20 outliers (Z-score > 3). Removing...
Cleaned shape: (980, 11)

=== CLASSIFICATION PIPELINE ===
Logistic (L2 Ridge) Best CV Accuracy: 0.8814 | Params: {'model__C': 0.1}
Logistic (L1 Lasso) Best C

In [5]:
import pandas as pd
import numpy as np
import os
from scipy import stats

from sklearn.datasets import make_regression, make_classification
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, ParameterGrid
from sklearn.preprocessing import StandardScaler

# Regression Models
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score

# Classification Models
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# ==========================================
# 1. UTILITY: GENERATE & READ DATA FILES
# ==========================================
def create_dummy_files():
    """Generates synthetic data and saves them to CSV, JSON, and Excel."""
    print("--- Generating Dummy Files ---")
    
    # Regression Data
    X_reg, y_reg = make_regression(n_samples=1000, n_features=10, noise=0.1, random_state=42)
    df_reg = pd.DataFrame(X_reg, columns=[f'feature_{i}' for i in range(10)])
    df_reg['target'] = y_reg
    df_reg.loc[10:15, 'feature_0'] = np.nan 
    df_reg = pd.concat([df_reg, df_reg.iloc[0:5]]) 
    df_reg.loc[20, 'feature_1'] = 9999.0 
    df_reg.to_csv("regression_data.csv", index=False)
    
    # Classification Data
    X_clf, y_clf = make_classification(n_samples=1000, n_features=10, random_state=42)
    df_clf = pd.DataFrame(X_clf, columns=[f'feature_{i}' for i in range(10)])
    df_clf['target'] = y_clf
    df_clf.to_json("classification_data.json", orient='records')

    print("Files created: regression_data.csv, classification_data.json\n")

def load_data(filepath):
    """Reads data dynamically based on the file extension."""
    ext = os.path.splitext(filepath)[1].lower()
    if ext == '.csv': return pd.read_csv(filepath)
    elif ext in ['.xlsx', '.xls']: return pd.read_excel(filepath)
    elif ext == '.json': return pd.read_json(filepath)
    else: raise ValueError(f"Unsupported file extension: {ext}")

# ==========================================
# 2. PREPROCESSING WORKFLOW
# ==========================================
def clean_data(df, target_col):
    """Handles duplicates, missing values, and outliers."""
    df = df.drop_duplicates()
    df = df.fillna(df.median())
    
    features = df.drop(columns=[target_col])
    z_scores = np.abs(stats.zscore(features))
    outlier_mask = (z_scores < 3).all(axis=1)
    
    df_cleaned = df[outlier_mask]
    return df_cleaned.drop(columns=[target_col]), df_cleaned[target_col]

# ==========================================
# 3. REGRESSION (Manual CV & Scaling)
# ==========================================
def run_regression_pipeline(X, y):
    print("=== REGRESSION (No Pipeline) ===")
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Convert to numpy arrays for easier fold indexing
    X_train_np, y_train_np = X_train.values, y_train.values
    X_test_np, y_test_np = X_test.values, y_test.values
    
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    
    # Notice we pass the Class (e.g., Ridge), not an instantiated object
    models = {
        "Ridge (L2)": (Ridge, {'alpha': [0.1, 1.0, 10.0]}),
        "Lasso (L1)": (Lasso, {'alpha': [0.1, 1.0, 10.0]}),
        "ElasticNet (L1+L2)": (ElasticNet, {'alpha': [0.1, 1.0], 'l1_ratio': [0.2, 0.5, 0.8]})
    }
    
    best_overall_score = float('inf')
    best_model_name = ""
    best_model_instance = None
    best_scaler_instance = None

    for name, (model_class, param_grid) in models.items():
        # 1. Outer Loop: Iterate through hyperparameter combinations
        for params in ParameterGrid(param_grid):
            fold_rmses = []
            
            # 2. Inner Loop: Iterate through Cross-Validation folds
            for train_idx, val_idx in kf.split(X_train_np):
                
                # Split fold data
                X_fold_train, X_fold_val = X_train_np[train_idx], X_train_np[val_idx]
                y_fold_train, y_fold_val = y_train_np[train_idx], y_train_np[val_idx]
                
                # SCALE INSIDE THE LOOP: Fit scaler ONLY on the training fold
                scaler = StandardScaler()
                X_fold_train_scaled = scaler.fit_transform(X_fold_train)
                X_fold_val_scaled = scaler.transform(X_fold_val)
                
                # Train model with current parameters
                model = model_class(**params)
                model.fit(X_fold_train_scaled, y_fold_train)
                
                # Evaluate fold
                y_fold_pred = model.predict(X_fold_val_scaled)
                fold_rmse = np.sqrt(mean_squared_error(y_fold_val, y_fold_pred))
                fold_rmses.append(fold_rmse)
                
            avg_rmse = np.mean(fold_rmses)
            
            # Check if this hyperparameter combo is the best so far
            if avg_rmse < best_overall_score:
                best_overall_score = avg_rmse
                best_model_name = name
                
                # Retrain the absolute best model on the FULL 80% training data
                best_scaler_instance = StandardScaler()
                X_train_full_scaled = best_scaler_instance.fit_transform(X_train_np)
                
                best_model_instance = model_class(**params)
                best_model_instance.fit(X_train_full_scaled, y_train_np)

    print(f"Best Model Found: {best_model_name} (CV RMSE: {best_overall_score:.4f})")
    
    # Final Evaluation on the 20% Holdout Test Set
    X_test_scaled = best_scaler_instance.transform(X_test_np)
    y_pred = best_model_instance.predict(X_test_scaled)
    print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test_np, y_pred)):.4f}\n")

# ==========================================
# 4. CLASSIFICATION (Manual CV & Scaling)
# ==========================================
def run_classification_pipeline(X, y):
    print("=== CLASSIFICATION (No Pipeline) ===")
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    
    X_train_np, y_train_np = X_train.values, y_train.values
    X_test_np, y_test_np = X_test.values, y_test.values
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    models = {
        "Logistic (L2 Ridge)": (LogisticRegression, {'solver': ['saga'], 'penalty': ['l2'], 'max_iter': [1000], 'C': [0.1, 1.0, 10.0]}),
        "Logistic (L1 Lasso)": (LogisticRegression, {'solver': ['saga'], 'penalty': ['l1'], 'max_iter': [1000], 'C': [0.1, 1.0, 10.0]}),
        "Logistic (ElasticNet)": (LogisticRegression, {'solver': ['saga'], 'penalty': ['elasticnet'], 'max_iter': [1000], 'C': [0.1, 1.0], 'l1_ratio': [0.2, 0.5, 0.8]})
    }
    
    best_overall_score = 0
    best_model_name = ""
    best_model_instance = None
    best_scaler_instance = None

    for name, (model_class, param_grid) in models.items():
        for params in ParameterGrid(param_grid):
            fold_accuracies = []
            
            for train_idx, val_idx in skf.split(X_train_np, y_train_np):
                
                X_fold_train, X_fold_val = X_train_np[train_idx], X_train_np[val_idx]
                y_fold_train, y_fold_val = y_train_np[train_idx], y_train_np[val_idx]
                
                # SCALE INSIDE THE LOOP
                scaler = StandardScaler()
                X_fold_train_scaled = scaler.fit_transform(X_fold_train)
                X_fold_val_scaled = scaler.transform(X_fold_val)
                
                model = model_class(**params)
                model.fit(X_fold_train_scaled, y_fold_train)
                
                y_fold_pred = model.predict(X_fold_val_scaled)
                fold_accuracies.append(accuracy_score(y_fold_val, y_fold_pred))
                
            avg_accuracy = np.mean(fold_accuracies)
            
            if avg_accuracy > best_overall_score:
                best_overall_score = avg_accuracy
                best_model_name = name
                
                # Retrain on full 80% train set
                best_scaler_instance = StandardScaler()
                X_train_full_scaled = best_scaler_instance.fit_transform(X_train_np)
                
                best_model_instance = model_class(**params)
                best_model_instance.fit(X_train_full_scaled, y_train_np)

    print(f"Best Model Found: {best_model_name} (CV Accuracy: {best_overall_score:.4f})")
    
    X_test_scaled = best_scaler_instance.transform(X_test_np)
    y_pred = best_model_instance.predict(X_test_scaled)
    print("--- Final Holdout Test Report ---")
    print(classification_report(y_test_np, y_pred))

# ==========================================
# 5. MAIN EXECUTION BLOCK
# ==========================================
if __name__ == "__main__":
    create_dummy_files()
    
    print("LOADING CSV FILE...")
    df_reg = load_data("regression_data.csv")
    X_reg, y_reg = clean_data(df_reg, target_col='target')
    run_regression_pipeline(X_reg, y_reg)
    
    print("LOADING JSON FILE...")
    df_clf = load_data("classification_data.json")
    X_clf, y_clf = clean_data(df_clf, target_col='target')
    run_classification_pipeline(X_clf, y_clf)

--- Generating Dummy Files ---
Files created: regression_data.csv, classification_data.json

LOADING CSV FILE...
=== REGRESSION (No Pipeline) ===
Best Model Found: Ridge (L2) (CV RMSE: 1.4461)
Test RMSE: 0.1810

LOADING JSON FILE...
=== CLASSIFICATION (No Pipeline) ===
Best Model Found: Logistic (L2 Ridge) (CV Accuracy: 0.8814)
--- Final Holdout Test Report ---
              precision    recall  f1-score   support

           0       0.81      0.87      0.84        99
           1       0.86      0.79      0.82        97

    accuracy                           0.83       196
   macro avg       0.83      0.83      0.83       196
weighted avg       0.83      0.83      0.83       196

